<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/FINAL_UNESCOTEXT_SARVAM_DELIVER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

# 1. Grab your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# 2. Initialize the API
api = HfApi()

# 3. Upload the file
api.upload_file(
    path_or_fileobj="/content/TEXTUNESCO.pdf",
    path_in_repo="TEXTUNESCO.pdf",
    repo_id="frankmorales2020/sarvam-30b-fp8-unesco-resilient",
    repo_type="model", # or "dataset"/"space" if applicable
    token=HF_TOKEN
)

print("File uploaded successfully!")

In [ ]:
!pip install codecarbon -q
!pip install vllm==0.19.1 -q
!pip install https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl -q
!pip uninstall -y protobuf
!pip install protobuf==5.26.1 -q

In [ ]:
#!pip uninstall -y transformers
#!pip install git+https://github.com/huggingface/transformers -q

In [2]:
!pip show transformers flash-attn vllm codecarbon huggingface_hub torch

Name: transformers
Version: 5.7.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: compressed-tensors, peft, sentence-transformers, vllm, xgrammar
---
Name: flash_attn
Version: 2.8.3
Summary: Flash Attention: Fast and Memory-Efficient Exact Attention
Home-page: https://github.com/Dao-AILab/flash-attention
Author: Tri Dao
Author-email: tri@tridao.me
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: einops, torc

## VLLM

SNAPSHOT

In [3]:
import os
from google.colab import userdata
from huggingface_hub import snapshot_download

# 1. Set the token from your Colab Secrets
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 2. Download the repository
# Using 'local_dir' ensures it's easy to find in your file browser
model_path = snapshot_download(
    repo_id="frankmorales2020/sarvam-30b-fp8-unesco-resilient",
    local_dir="./sarvam-30b",
    token=os.environ['HF_TOKEN']
)

print(f"Model successfully downloaded to: {model_path}")

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Model successfully downloaded to: /content/sarvam-30b


In [4]:
!rm -rf /root/.cache/
!rm -rf /content/sarvam-30b/*.safetensors

In [5]:
!cat /content/sarvam-30b/vllm_config.yaml

model: frankmorales2020/sarvam-30b-fp8-unesco-resilient
tokenizer: frankmorales2020/sarvam-30b-fp8-unesco-resilient
trust_remote_code: true
dtype: bfloat16
quantization: compressed-tensors
kv_cache_dtype: fp8
block_size: 16
gpu_memory_utilization: 0.90
max_model_len: 65536
max_num_seqs: 64
enforce_eager: true
served_model_name: sarvam-30b


In [ ]:
import os
from google.colab import userdata

# 1. Authentication for your private repo
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 2. Performance & Stability Flags
# Disable the version check to avoid strict CUDA/FlashInfer mismatch errors
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
# Disable the MoE FP8 kernel that can cause hangs with Sarvam/Mixtral architectures
os.environ['VLLM_USE_FLASHINFER_MOE_FP8'] = '0'

# 3. Cleanup TensorFlow noise (Colab has TF pre-installed)
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# 4. Launch the server
# We use !vllm, and it will inherit the os.environ variables set above
!vllm serve --config /content/sarvam-30b/vllm_config.yaml

(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299] 
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.19.1
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299]   █▄█▀ █     █     █     █  model   frankmorales2020/sarvam-30b-fp8-unesco-resilient
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:299] 
(APIServer pid=2702) INFO 04-30 14:59:46 [utils.py:233] non-default args: {'model': 'frankmorales2020/sarvam-30b-fp8-unesco-resilient', 'tokenizer': 'frankmorales2020/sarvam-30b-fp8-unesco-resilient', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 65536, 'quantization': 'compressed-tensors', 'enforce_eager': True, 'served_model_name': ['sarvam-30b'], 'block_size': 16, 'kv_cache_dtype': 'fp8', 'max_num_seqs': 64}
config.json: 2.74kB [00:00, 3.20MB/s]
configur